# Segger Losses (v0.2.0)

This notebook explains how Segger v0.2.0 builds its model and computes each loss.
It also shows how the optional alignment loss uses scRNA-seq references to further
clean up contamination.


## Requirements

- Segger installed (GPU-only; requires RAPIDS: CuPy/cuDF/cuML/cuGraph/cuSpatial).
- A spatial dataset in raw platform format or SpatialData `.zarr`.
- Optional: an scRNA-seq reference `.h5ad` with cell type annotations.


## Model overview (what Segger is doing)

Segger frames cell segmentation as **link prediction** on a heterogeneous graph:

- **Nodes**: transcripts (`tx`) and boundaries (`bd`)
- **Edges**:
  - `tx -> neighbors -> tx`: transcript-transcript neighbors (kNN within distance)
  - `tx -> belongs -> bd`: training edges from known segmentation
  - `tx -> neighbors -> bd`: candidate edges for inference

The model produces L2-normalized embeddings for `tx` and `bd`, and predicts
assignments using cosine similarity.


## Graph construction details

Segger constructs three edge types:

1. **Transcript-transcript neighbors** using KD-tree kNN with a distance cutoff.
2. **Transcript-boundary belongs** edges from the training segmentation mask.
3. **Transcript-boundary prediction** edges by testing inclusion in scaled polygons.

For prediction, Segger scores candidate `tx->bd` edges and assigns each transcript
to the best-scoring boundary if it clears a threshold (fixed or per-gene).


## Node features and embeddings

- **Transcript features** come from a learned gene embedding table. The initial
  gene embedding is computed by PCA on a gene-gene correlation matrix built from
  normalized counts in the spatial data (see `data/utils/anndata.py`).

- **Boundary features** can be either:
  - Expression-based PCA (`cells_representation_mode = pca`), or
  - Morphology-based features (`cells_representation_mode = morphology`).

- **Morphology features** (when enabled) include area, convexity, elongation,
  and circularity, then projected to the embedding space.

- **Positional embeddings** are added using sinusoidal encodings of coordinates.

All embeddings are L2-normalized so dot products equal cosine similarity.


## GNN architecture (ISTEncoder)

The encoder uses a SkipGAT stack over the heterogeneous graph:

1. Linear projection of node features
2. Positional embedding addition
3. Multiple SkipGAT (GATv2) layers
4. Final projection + L2 normalization

Output embeddings are used directly by the loss functions and inference scoring.


## Clustering and phenograph (why it matters for losses)

Segger builds an AnnData object from transcripts and then computes clusters for
cells and genes:

- **Cell embeddings**: PCA on normalized counts (`cuml.PCA`).
- **Gene embeddings**: PCA on the gene-gene correlation matrix.
- **Clustering**: `phenograph_rapids` (kNN -> Jaccard graph -> Louvain) on GPU.

These clusters provide a **similarity matrix** between clusters that drives the
triplet sampling strategy used by transcript and boundary losses.


## Loss overview and weight scheduling

Segger uses a multi-task objective:

$$\mathcal{L} = w_{tx} \mathcal{L}_{tx} + w_{bd} \mathcal{L}_{bd} + w_{sg} \mathcal{L}_{sg}$$

**Why multiple losses**:
- `tx` loss stabilizes transcript embeddings using gene cluster structure.
- `bd` loss stabilizes boundary embeddings using cell cluster structure.
- `sg` loss ties transcripts to boundaries for the actual segmentation task.

Loss weights are cosine-scheduled from start values to end values over training.
Alignment loss is optional and is combined with the main loss by a stable default
schedule.


## Loss 1: Transcript loss (TripletLoss)

**Purpose**: Encourage transcript embeddings from similar gene clusters to be close.

Triplets are sampled with `FastTripletSelector` using the gene cluster similarity
matrix. Positives come from similar clusters, negatives from dissimilar clusters.

Triplet margin loss:

$$\mathcal{L}_{triplet} = \max(0, ||a - p||^2 - ||a - n||^2 + m)$$

Where the anchor `a` is a transcript embedding, `p` a positive transcript, and
`n` a negative transcript.


## Loss 2: Boundary loss (MetricLoss)

**Purpose**: Keep boundaries with similar expression profiles close in embedding
space.

The same triplet sampling strategy is used, but the objective is to match cosine
similarity to the cluster distance:

$$\mathcal{L}_{metric} = 	ext{MSE}(\cos(a,p), 1-d_{ap}) + 	ext{MSE}(\cos(a,n), 1-d_{an})$$

Here `d_{ap}` and `d_{an}` come from the cluster similarity matrix.


## Loss 3: Segmentation loss (Triplet or BCE)

**Purpose**: Learn transcript-to-boundary assignment.

Triplet form (default):

$$\mathcal{L}_{sg} = \max(0, ||h_t - h_{b^+}||^2 - ||h_t - h_{b^-}||^2 + m)$$

- `b^+` is the true boundary (from the training mask).
- `b^-` is a randomly sampled boundary.

BCE form (optional):

$$\mathcal{L}_{BCE} = -[y \log \sigma(s) + (1-y) \log(1-\sigma(s))]$$

Use `--segmentation-loss bce` if you want the BCE baseline from v1.


## How Segger computes the losses (code-level summary)

Inside `models/lightning_model.py` (`LitISTEncoder.get_losses`):

1. **Forward pass** produces embeddings for `tx` and `bd`.
2. **Transcript loss** uses `tx` embeddings and gene cluster labels.
3. **Boundary loss** uses `bd` embeddings and cell cluster labels.
4. **Segmentation loss** uses `tx->bd` edges from the training mask and random
   negative boundaries.
5. **Alignment loss** is computed only if ME alignment edges exist in the batch.
6. **Weight scheduling** combines the losses into the final objective.

This order is important because alignment loss is computed on top of a stabilized
embedding space from the other losses.


## Inference scoring (after training)

Segger assigns each transcript to the best candidate boundary by cosine similarity.
Assignments can be filtered by:

- A **fixed minimum similarity** threshold, or
- A **per-gene automatic threshold** (Li + Yen methods).

Unassigned transcripts can optionally be grouped using fragment mode, which builds
connected components over the tx-tx graph.


## Alignment loss (optional cleanup)

Alignment loss adds biological constraints using **mutually exclusive (ME) genes**
derived from scRNA-seq references. It operates only on a subset of tx-tx neighbor
edges:

- **Positive edges**: same-gene neighbor pairs
- **Negative edges**: ME gene pairs

Other tx-tx edges are ignored. Positives are capped to at most 3x negatives.

Contrastive form with fixed margin `m = 0.2`:

$$\mathcal{L}_{align} = E_{pos}[(1-s)^2] + E_{neg}[\max(0, s-m)^2]$$

This encourages ME markers to separate in embedding space, reducing cross-cell
mixing and contamination artifacts.


## Why alignment loss helps contamination

Many contamination patterns look like small numbers of transcripts from a cell-type
specific marker appearing inside a different cell. If those markers are mutually
exclusive in scRNA-seq data, alignment loss explicitly penalizes those pairings in
the embedding space. This reduces false merges and improves assignments, especially
in crowded tissue or weak boundary regions.


## ME gene discovery from scRNA-seq (details)

Segger computes ME gene pairs from a reference `.h5ad` in `validation/me_genes.py`:

1. Load reference and subsample to at most 1000 cells per cell type.
2. Normalize and log1p if needed.
3. Find positive/negative marker sets per cell type using percentile cutoffs.
4. Mark a gene as exclusive if it is high in its own cell type and low elsewhere.
5. Build ME gene pairs by combining exclusive markers from different cell types.

ME pairs are cached next to the `.h5ad` for reuse. Gene names must match between
the reference and spatial data.


## Hard-coded or internal behaviors (v0.2.0)

These are fixed unless you change code:

- Alignment margin is fixed at 0.2.
- Alignment positives are capped at 3x the number of negatives.
- ME thresholds are fixed (pos/neg percentiles = 10, min percent expressing = 30,
  expr_in > 0.25, expr_out < 0.03).
- ME discovery subsamples to at most 1000 cells per cell type.
- ME pairs are matched against `adata.var_names` (gene naming must align).


## User-settable knobs (common)

You can tune these from the CLI if needed:

- `--segmentation-loss` (triplet or bce)
- `--transcripts-margin`, `--segmentation-margin`
- `--transcripts-loss-weight-start/end`, `--cells-loss-weight-start/end`,
  `--segmentation-loss-weight-start/end`
- `--alignment-loss` (enable alignment loss)
- `--scrna-reference-path`, `--scrna-celltype-column`
- Representation: `--cells-representation-mode` (pca or morphology)
- Graph construction: `--transcripts-graph-max-k`, `--transcripts-graph-max-dist`,
  `--prediction-graph-max-k`, `--prediction-graph-scale-factor`
- Clustering: `--cells-clusters-n-neighbors`, `--cells-clusters-resolution`,
  `--genes-clusters-n-neighbors`, `--genes-clusters-resolution`


## Recommended runs (stable defaults)

Baseline (no alignment loss):

```bash
segger segment -i /path/to/data -o /path/to/out
```

Best default when you have a reference (stable and general):

```bash
segger segment -i /path/to/data -o /path/to/out   --alignment-loss   --scrna-reference-path /path/to/reference.h5ad   --scrna-celltype-column celltype
```

This uses Segger's default alignment schedule and loss combination, which is stable
for most datasets.


In [ ]:
# Optional: compute ME gene pairs directly (requires scanpy + reference.h5ad)
from pathlib import Path

try:
    from segger.validation.me_genes import load_me_genes_from_scrna
except Exception as e:
    print(f"Segger not importable in this environment: {e}")
    load_me_genes_from_scrna = None

scrna_path = Path("reference.h5ad")
celltype_col = "celltype"

if load_me_genes_from_scrna and scrna_path.exists():
    me_pairs, markers = load_me_genes_from_scrna(
        scrna_path=scrna_path,
        cell_type_column=celltype_col,
    )
    print(f"ME gene pairs: {len(me_pairs)}")
    print("Example pairs:", me_pairs[:10])
else:
    print("Provide reference.h5ad and a valid celltype column to compute ME pairs.")


In [ ]:
# Optional: toy loss computation on random data (requires torch)
try:
    import torch
    from segger.models.triplet_loss import TripletLoss, MetricLoss
    from segger.models.alignment_loss import AlignmentLoss

    # Dummy embeddings and labels
    embeddings = torch.randn(100, 32)
    labels = torch.randint(0, 5, (100,))
    cluster_sim = torch.eye(5)

    loss_tx = TripletLoss(cluster_sim, margin=0.3)
    loss_bd = MetricLoss(cluster_sim)
    print("Triplet loss:", loss_tx(embeddings, labels).item())
    print("Metric loss:", loss_bd(embeddings, labels).item())

    # Alignment loss on random pairs
    align = AlignmentLoss()
    src = embeddings[:50]
    dst = embeddings[50:100]
    labels_align = torch.randint(0, 2, (50,))
    print("Alignment loss:", align(src, dst, labels_align).item())
except Exception as e:
    print(f"Skipping toy loss demo: {e}")


In [ ]:
# Optional: contamination estimates (requires cupy/cuml and compatible outputs)
from pathlib import Path

try:
    import scanpy as sc
    from segger.validation.contamination import (
        expression_summary_from_anndata,
        calculate_contamination,
    )
except Exception as e:
    print(f"Missing dependencies for contamination analysis: {e}")
    sc = None

segger_out = Path("output/anndata.h5ad")
scrna_ref = Path("reference.h5ad")

if sc and segger_out.exists() and scrna_ref.exists():
    adata = sc.read_h5ad(segger_out)
    ref = sc.read_h5ad(scrna_ref)

    # Build reference summary (adjust raw_layer + cell_type_col as needed)
    ref_summary = expression_summary_from_anndata(
        ref,
        cell_type_col="celltype",
        raw_layer="counts",
    )

    calculate_contamination(
        adata,
        ref_summary,
        counts_layer="counts",
        spatial_key="spatial",
        cell_type_key="cell_type",
    )

    adata.write_h5ad("output/anndata_with_contamination.h5ad")
    print("Wrote contamination-augmented AnnData.")
else:
    print("Set paths and layer/column names to run contamination analysis.")


## Monitoring and diagnostics

During training, Segger logs:

- `train:loss_tx`, `train:loss_bd`, `train:loss_sg`
- `train:loss_align` (only if alignment loss is enabled)

Expect all losses to decrease. If alignment loss stays high, it usually means
ME pairs are too strict or gene name matching is poor.


## Common gotchas

- **Gene name mismatch** between scRNA-seq and spatial data is the #1 cause of
  missing ME pairs.
- **No ME pairs** means alignment loss is effectively disabled.
- **Unstable training**: try fewer edges per batch or smaller tiles.
- **BCE mode** is useful for debugging if triplet loss fails to converge.
